# WPT-XLSR-AASIST × Fake-or-Real(FoR) 20,000개 학습

이 노트북은 Google Colab에서 다음 작업을 순서대로 수행합니다.

1. Kaggle의 [The Fake-or-Real (FoR) Dataset](https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset)을 코드로 내려받습니다.
2. `for-2sec`의 공식 `training`/`validation` 폴더를 사용하고, 학습 데이터는 최대 20,000개(FAKE/REAL 균형 표집)만 사용합니다.
3. 공식 [All-Type-ADD](https://github.com/xieyuankun/All-Type-ADD) 저장소의 고정 커밋과 `facebook/wav2vec2-xls-r-300m`으로 WPT-XLSR-AASIST를 구성합니다.
4. 매 epoch 재시작용 `last_checkpoint.pt`와 검증 EER 기준 최적 가중치 `best.pt`를 Google Drive에 저장합니다.
5. 마지막 단계에서 공식 Dacon baseline 자산과 `best.pt`를 결합해 바로 제출할 `submit.zip`을 생성합니다.

**중요한 호환 규약**

- 제출 추론 코드는 `softmax(logits)[:, 0]`을 FAKE 확률로 사용합니다. 따라서 이 노트북은 의도적으로 `FAKE=0`, `REAL=1`로 학습합니다.
- 학습과 제출 추론 모두 `num_prompt_tokens=6`, `num_wavelet_tokens=4`, `prompt_dim=1024`를 고정합니다.
- `best.pt`는 `torch.load(..., weights_only=True)`로 바로 읽을 수 있는 순수 `state_dict`입니다.

**실행 전:** Colab 메뉴에서 `런타임 → 런타임 유형 변경 → GPU`를 선택하세요. WPT-XLSR-AASIST는 XLS-R 300M을 포함하므로 CPU 학습은 현실적으로 어렵습니다. KaggleHub가 폴더 단위 다운로드를 지원하지 않는 버전에서는 전체 데이터셋(약 20.2GB)을 받은 뒤 `for-2sec`만 학습에 사용합니다.


In [ ]:
# Colab 기본 PyTorch/torchaudio는 그대로 사용하고, 공식 코드에 필요한 패키지만 설치합니다.
%pip install -q --upgrade \
    "kagglehub>=1.0.2" \
    "transformers==4.36.2" \
    "pytorch-wavelets==1.3.0" \
    "soundfile>=0.12.1" \
    "scipy>=1.10" \
    "scikit-learn>=1.3" \
    "pandas>=2.0" \
    "tqdm>=4.66" \
    "safetensors>=0.4"


In [ ]:
from pathlib import Path
from collections import Counter
from contextlib import nullcontext
import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from scipy.signal import resample_poly
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ── 사용자가 주로 조정할 설정 ──────────────────────────────────────────────
SEED = 688
MAX_TRAIN_SAMPLES = 20_000       # 요청한 학습 상한
MAX_VAL_SAMPLES = 2_000          # 별도 검증 표본 상한(학습 20,000개에 포함되지 않음)
EPOCHS = 5
BATCH_SIZE = 2                   # T4 16GB의 안전한 시작값; OOM이면 1로 낮추세요.
GRAD_ACCUM_STEPS = 8             # 유효 배치 크기 = BATCH_SIZE × 이 값
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
MAX_GRAD_NORM = 5.0
WARMUP_RATIO = 0.05
EARLY_STOPPING_PATIENCE = 3
NUM_WORKERS = 2
USE_AMP = True
RESUME = True                    # last_checkpoint.pt가 있으면 이어서 학습
USE_GOOGLE_DRIVE = True
OUTPUT_DIR_NAME = "WPT_XLSR_AASIST_FoR_20000"

# ── 제출 생성기 호환을 위해 바꾸지 말아야 할 설정 ─────────────────────────
TARGET_SR = 16_000
TARGET_NUM_SAMPLES = 64_600
PROMPT_DIM = 1_024
NUM_PROMPT_TOKENS = 6
NUM_WAVELET_TOKENS = 4
PROMPT_DROPOUT = 0.1             # 추론 시 eval()에서 비활성화되므로 0.0 로더와 호환
CLASS_TO_INDEX = {"fake": 0, "real": 1}

KAGGLE_DATASET = "mohammedabdeldayem/the-fake-or-real-dataset"
DATASET_VARIANT = "for-2sec"
WPT_REPO = "https://github.com/xieyuankun/All-Type-ADD.git"
WPT_COMMIT = "4faae8ba700aa93f03cfc87c100dc626b6a7e68b"
XLSR_REPO = "facebook/wav2vec2-xls-r-300m"

assert MAX_TRAIN_SAMPLES <= 20_000
assert CLASS_TO_INDEX == {"fake": 0, "real": 1}
assert (PROMPT_DIM, NUM_PROMPT_TOKENS, NUM_WAVELET_TOKENS) == (1024, 6, 4)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
print("PyTorch:", torch.__version__)
print("GPU:", GPU_NAME)
print("라벨 규약:", CLASS_TO_INDEX, "(class 0 = FAKE)")


In [ ]:
# 체크포인트는 런타임이 종료되어도 남도록 Google Drive에 저장합니다.
IN_COLAB = "google.colab" in sys.modules

if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("USE_GOOGLE_DRIVE=True는 Colab에서 실행해야 합니다.")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = Path("/content/drive/MyDrive") / OUTPUT_DIR_NAME
else:
    OUTPUT_DIR = Path("/content") / OUTPUT_DIR_NAME

WORK_DIR = Path("/content/wpt_xlsr_aasist_work")
REPO_DIR = WORK_DIR / "All-Type-ADD"
XLSR_DIR = WORK_DIR / "xlsr_300m"
BEST_PATH = OUTPUT_DIR / "best.pt"
LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
CONFIG_PATH = OUTPUT_DIR / "training_config.json"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage("/content").free / 2**30
print("작업 폴더:", WORK_DIR)
print("결과 폴더:", OUTPUT_DIR)
print(f"Colab 남은 디스크: {free_gb:.1f} GiB")
if free_gb < 35:
    print("경고: KaggleHub가 전체 데이터셋을 받아야 하면 디스크가 부족할 수 있습니다.")


## 1. Kaggle에서 FoR 데이터셋 불러오기

공개 데이터셋은 일반적으로 로그인 없이 받을 수 있습니다. 인증 오류가 발생할 때만 `kagglehub.login()` 입력창이 나타나며, Kaggle 설정의 API 토큰을 붙여 넣으면 됩니다. 별도로 데이터셋 파일을 내려받아 Colab에 업로드할 필요는 없습니다.


In [ ]:
# Colab의 KaggleHub 캐시 위치를 런타임 디스크로 고정합니다.
os.environ.setdefault("KAGGLEHUB_CACHE", str(WORK_DIR / "kagglehub_cache"))
os.environ["DISABLE_COLAB_CACHE"] = "1"
import kagglehub

def _download_for_dataset():
    # 가능하면 for-2sec 폴더만, 미지원 버전이면 전체 데이터셋을 다운로드합니다.
    try:
        path = kagglehub.dataset_download(KAGGLE_DATASET, path=DATASET_VARIANT)
        print("for-2sec 폴더만 다운로드했습니다.")
        return Path(path)
    except Exception as folder_error:
        print("폴더 단위 다운로드를 사용할 수 없어 전체 데이터셋으로 전환합니다.")
        print("폴더 요청 메시지:", type(folder_error).__name__, str(folder_error)[:300])
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))

try:
    DATA_ROOT = _download_for_dataset()
except Exception as first_error:
    print("Kaggle 인증이 필요하거나 다운로드가 중단되었습니다:", repr(first_error))
    print("Kaggle API 토큰을 입력한 뒤 자동으로 다시 시도합니다.")
    kagglehub.login()
    DATA_ROOT = _download_for_dataset()

if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
print("Kaggle 데이터 위치:", DATA_ROOT)


## 2. `for-2sec` 인덱스 생성 및 균형 표집

제공된 `training`과 `validation` 분할을 그대로 유지합니다. 학습 표본은 가능한 경우 FAKE 10,000개 + REAL 10,000개로 정확히 20,000개를 구성합니다. 데이터가 부족한 경우에는 두 클래스에서 가능한 동일 개수만 사용하므로 항상 설정값 이하입니다.


In [ ]:
AUDIO_EXTENSIONS = {".wav", ".flac", ".ogg"}
SPLIT_ALIASES = {
    "training": "train", "train": "train",
    "validation": "val", "valid": "val", "dev": "val",
    "testing": "test", "test": "test",
}

def parse_for_path(path):
    lower_parts = [part.lower() for part in path.parts]
    # 전체 데이터셋을 받은 경우 다른 세 버전이 섞이지 않도록 제한합니다.
    if not any(part in {"for-2sec", "for-2seconds"} for part in lower_parts):
        return None

    split = next((SPLIT_ALIASES[p] for p in lower_parts if p in SPLIT_ALIASES), None)
    label = "fake" if "fake" in lower_parts else ("real" if "real" in lower_parts else None)
    if split is None or label is None:
        return None
    return {"path": str(path), "split": split, "label_name": label,
            "label": CLASS_TO_INDEX[label]}

audio_paths = [p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS]
records = [record for path in audio_paths if (record := parse_for_path(path)) is not None]
all_df = pd.DataFrame(records)

if all_df.empty:
    raise RuntimeError(
        f"{DATA_ROOT} 아래에서 for-2sec/training|validation/fake|real 구조를 찾지 못했습니다."
    )

print("발견된 for-2sec 파일 수:", f"{len(all_df):,}")
display(all_df.groupby(["split", "label_name"]).size().rename("count").to_frame())

def balanced_sample(frame, split, limit, seed):
    part = frame[frame["split"] == split].copy()
    counts = part.groupby("label_name").size()
    if not {"fake", "real"}.issubset(counts.index):
        raise RuntimeError(f"{split}에 fake/real 두 클래스가 모두 필요합니다: {counts.to_dict()}")
    per_class = min(limit // 2, int(counts.loc["fake"]), int(counts.loc["real"]))
    if per_class < 1:
        raise RuntimeError(f"{split} 표본이 부족합니다: {counts.to_dict()}")
    chunks = [
        part[part["label_name"] == name].sample(n=per_class, random_state=seed)
        for name in ("fake", "real")
    ]
    sampled = pd.concat(chunks, ignore_index=True)
    return sampled.sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_df = balanced_sample(all_df, "train", MAX_TRAIN_SAMPLES, SEED)
val_df = balanced_sample(all_df, "val", MAX_VAL_SAMPLES, SEED + 1)

assert len(train_df) <= 20_000
assert set(train_df["label"].unique()) == {0, 1}
assert set(val_df["label"].unique()) == {0, 1}

print("학습 표본:", len(train_df), train_df["label_name"].value_counts().to_dict())
print("검증 표본:", len(val_df), val_df["label_name"].value_counts().to_dict())

# 실행 기록용 manifest. 파일 경로는 현재 Colab 런타임 기준입니다.
train_df.to_csv(OUTPUT_DIR / "train_manifest.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_manifest.csv", index=False)


In [ ]:
class FoRAudioDataset(Dataset):
    def __init__(self, frame, training=False):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.training = training

    def __len__(self):
        return len(self.paths)

    def _load_audio(self, path):
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
        if sample_rate != TARGET_SR:
            divisor = math.gcd(int(sample_rate), TARGET_SR)
            audio = resample_poly(audio, TARGET_SR // divisor, int(sample_rate) // divisor)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if audio.size == 0:
            raise ValueError(f"빈 오디오: {path}")
        return audio

    def _fix_length(self, audio):
        if audio.size < TARGET_NUM_SAMPLES:
            repeats = math.ceil(TARGET_NUM_SAMPLES / audio.size)
            audio = np.tile(audio, repeats)
        max_start = audio.size - TARGET_NUM_SAMPLES
        start = random.randint(0, max_start) if self.training and max_start > 0 else 0
        audio = audio[start:start + TARGET_NUM_SAMPLES]
        # 공식 데이터셋 코드의 파형 정규화와 동일한 형태
        audio = (audio - audio.mean()) / np.sqrt(audio.var() + 1e-7)
        return np.ascontiguousarray(audio, dtype=np.float32)

    def __getitem__(self, index):
        path = self.paths[index]
        audio = self._fix_length(self._load_audio(path))
        return torch.from_numpy(audio), Path(path).name, self.labels[index]

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
    persistent_workers=NUM_WORKERS > 0,
)

train_dataset = FoRAudioDataset(train_df, training=True)
val_dataset = FoRAudioDataset(val_df, training=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        drop_last=False, **loader_kwargs)

sanity_audio, sanity_names, sanity_labels = next(iter(train_loader))
assert sanity_audio.shape[1] == TARGET_NUM_SAMPLES
print("배치 파형:", tuple(sanity_audio.shape), sanity_audio.dtype)
print("배치 라벨:", sanity_labels.tolist(), "(0=FAKE, 1=REAL)")
print("예시 파일:", sanity_names[:2])


## 3. 공식 WPT 코드와 XLS-R 300M 준비

재현성을 위해 제출 패키지 생성기와 같은 Git 커밋을 checkout합니다. 원본 코드에서 현재 학습에 필요 없는 `torchvision`/요약 모듈 import와 매 배치마다 출력되는 attention/shape 로그만 제거합니다. 모델 구조와 `state_dict` 키는 바꾸지 않습니다.


In [ ]:
def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if not (REPO_DIR / ".git").is_dir():
    run(["git", "clone", WPT_REPO, str(REPO_DIR)])
run(["git", "checkout", "--detach", WPT_COMMIT], cwd=REPO_DIR)
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
assert actual_commit == WPT_COMMIT

# 불필요한 선택 의존성과 과도한 로그를 제거하는 idempotent 패치
model_py = REPO_DIR / "model.py"
text = model_py.read_text(encoding="utf-8")
text = text.replace("from pytorch_model_summary import summary\n", "")
text = text.replace("import torchvision.models as models\n", "")
model_py.write_text(text, encoding="utf-8")

exp_py = REPO_DIR / "exp" / "feature_extraction_exp.py"
text = exp_py.read_text(encoding="utf-8")
text = text.replace("self.model.config.output_attentions = True",
                    "self.model.config.output_attentions = False")
text = text.replace("            print(hidden_state.shape,'hidden_state')\n", "")
# pytorch_wavelets의 DWT 필터는 FP32이므로, 외부 AMP가 켜져 있어도
# 웨이블릿 forward/backward만 FP32로 유지합니다.
wavelet_call = "        LL, band = self.dwt(x)"
wavelet_amp_safe = (
    "        with torch.autocast(device_type=x.device.type, enabled=False):\n"
    "            LL, band = self.dwt(x.float())"
)
if wavelet_call in text:
    text = text.replace(wavelet_call, wavelet_amp_safe, 1)
elif wavelet_amp_safe not in text:
    raise RuntimeError("WaveletBlock의 DWT 호출부를 찾지 못했습니다.")
exp_py.write_text(text, encoding="utf-8")

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id=XLSR_REPO,
    local_dir=XLSR_DIR,
    ignore_patterns=["*.h5", "*.msgpack", "*.onnx", "*.ot"],
)
if not (XLSR_DIR / "config.json").is_file():
    raise FileNotFoundError("XLS-R config.json 다운로드 실패")
if not any((XLSR_DIR / name).is_file() for name in ("model.safetensors", "pytorch_model.bin")):
    raise FileNotFoundError("XLS-R 모델 가중치 다운로드 실패")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# 셀을 재실행했을 때 패치된 모듈을 다시 읽습니다.
for module_name in ["model", "feature_extraction", "exp.feature_extraction_exp"]:
    sys.modules.pop(module_name, None)
from model import WPTW2V2AASIST

print("WPT commit:", actual_commit)
print("XLS-R 위치:", XLSR_DIR)


In [ ]:
model = WPTW2V2AASIST(
    model_dir=str(XLSR_DIR),
    prompt_dim=PROMPT_DIM,
    device=DEVICE.type,
    sampling_rate=TARGET_SR,
    num_prompt_tokens=NUM_PROMPT_TOKENS,
    num_wavelet_tokens=NUM_WAVELET_TOKENS,
    dropout=PROMPT_DROPOUT,
    visual=False,
).to(DEVICE)

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
total_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(parameter.numel() for parameter in trainable_parameters)
frozen_xlsr_count = sum(
    parameter.numel() for parameter in model.wav2vec2_with_prompt.model.parameters()
    if not parameter.requires_grad
)
if frozen_xlsr_count == 0:
    raise RuntimeError("XLS-R backbone이 동결되지 않았습니다.")

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

# AMP + pytorch_wavelets 역전파 dtype을 짧게 사전 검사합니다.
amp_probe = torch.randn(
    1, NUM_WAVELET_TOKENS, PROMPT_DIM, device=DEVICE, requires_grad=True
)
with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    amp_probe_output = model.wav2vec2_with_prompt.wavelet_block(amp_probe)
    amp_probe_loss = amp_probe_output.float().mean()
amp_probe_loss.backward()
if amp_probe.grad is None or not torch.isfinite(amp_probe.grad).all():
    raise RuntimeError("WaveletBlock AMP 역전파 사전 검사가 실패했습니다.")
del amp_probe, amp_probe_output, amp_probe_loss
model.zero_grad(set_to_none=True)
print("WaveletBlock AMP forward/backward: OK (DWT는 FP32 고정)")

# 공식 WPT 전처리기가 내부에서 파형을 GPU로 옮기므로 파형 배치는 CPU로 전달합니다.
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    _, sanity_logits = model(sanity_audio[:1])
assert sanity_logits.shape == (1, 2)

print(f"전체 파라미터: {total_count / 1e6:.1f}M")
print(f"학습 파라미터: {trainable_count / 1e6:.1f}M")
print("sanity logits:", sanity_logits.float().cpu().numpy())
del sanity_logits
torch.cuda.empty_cache()


## 4. 학습·검증·체크포인트

- `best.pt`: 검증 EER가 가장 낮은 epoch의 순수 모델 `state_dict` (제출 생성기에 사용)
- `last_checkpoint.pt`: 모델 + 옵티마이저 + 스케줄러 + AMP + epoch 상태 (중단 후 재개용)
- `training_history.json`: epoch별 loss, accuracy, ROC-AUC, EER

`RESUME=True`이고 같은 Drive 폴더에 `last_checkpoint.pt`가 있으면 자동으로 다음 epoch부터 이어서 실행합니다.


In [ ]:
optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_optimizer_steps = max(1, EPOCHS * optimizer_steps_per_epoch)
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)

def lr_multiplier(step):
    if warmup_steps > 0 and step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = (step - warmup_steps) / max(1, total_optimizer_steps - warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def atomic_torch_save(obj, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary)
    os.replace(temporary, path)

def compute_metrics(labels, fake_scores, predictions, mean_loss):
    labels = np.asarray(labels, dtype=np.int64)
    fake_targets = (labels == CLASS_TO_INDEX["fake"]).astype(np.int64)
    fake_scores = np.asarray(fake_scores, dtype=np.float64)
    fpr, tpr, _ = roc_curve(fake_targets, fake_scores)
    fnr = 1.0 - tpr
    index = int(np.nanargmin(np.abs(fnr - fpr)))
    eer = float((fpr[index] + fnr[index]) / 2.0)
    return {
        "loss": float(mean_loss),
        "accuracy": float(accuracy_score(labels, predictions)),
        "roc_auc_fake": float(roc_auc_score(fake_targets, fake_scores)),
        "eer": eer,
    }

def train_one_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    total_items = 0
    correct = 0
    progress = tqdm(train_loader, desc=f"train {epoch + 1}/{EPOCHS}")

    for batch_index, (audio, _, labels) in enumerate(progress):
        labels = labels.to(DEVICE, non_blocking=True)
        group_start = (batch_index // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_size = min(GRAD_ACCUM_STEPS, len(train_loader) - group_start)

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, logits = model(audio)  # audio는 CPU; WPT 전처리기가 내부에서 이동
            raw_loss = criterion(logits, labels)
            scaled_loss = raw_loss / group_size

        scaler.scale(scaled_loss).backward()
        should_step = ((batch_index + 1) % GRAD_ACCUM_STEPS == 0) or (batch_index + 1 == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        batch_size = labels.size(0)
        total_loss += raw_loss.detach().item() * batch_size
        total_items += batch_size
        correct += (logits.detach().argmax(dim=1) == labels).sum().item()
        if batch_index % 50 == 0:
            progress.set_postfix(
                loss=f"{total_loss / total_items:.4f}",
                acc=f"{correct / total_items:.3f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

    return {"loss": total_loss / total_items, "accuracy": correct / total_items}

@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0
    total_items = 0
    labels_all, predictions_all, fake_scores_all = [], [], []
    progress = tqdm(val_loader, desc="validation")

    for audio, _, labels in progress:
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, logits = model(audio)
            loss = criterion(logits, labels)
        probabilities = torch.softmax(logits.float(), dim=1)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size
        labels_all.extend(labels.cpu().tolist())
        predictions_all.extend(logits.argmax(dim=1).cpu().tolist())
        fake_scores_all.extend(probabilities[:, CLASS_TO_INDEX["fake"]].cpu().tolist())

    return compute_metrics(
        labels_all,
        fake_scores_all,
        predictions_all,
        total_loss / total_items,
    )

start_epoch = 0
best_eer = float("inf")
epochs_without_improvement = 0
history = []

if RESUME and LAST_CHECKPOINT_PATH.is_file():
    print("재시작 체크포인트 로드:", LAST_CHECKPOINT_PATH)
    try:
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_eer = float(checkpoint["best_eer"])
    epochs_without_improvement = int(checkpoint.get("epochs_without_improvement", 0))
    history = list(checkpoint.get("history", []))
    del checkpoint
    gc.collect()
    print(f"epoch {start_epoch + 1}부터 재개, best EER={best_eer:.6f}")


In [ ]:
run_config = {
    "dataset": KAGGLE_DATASET,
    "dataset_variant": DATASET_VARIANT,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "actual_train_samples": len(train_dataset),
    "actual_val_samples": len(val_dataset),
    "class_to_index": CLASS_TO_INDEX,
    "target_sample_rate": TARGET_SR,
    "target_num_samples": TARGET_NUM_SAMPLES,
    "wpt_repository": WPT_REPO,
    "wpt_commit": WPT_COMMIT,
    "xlsr_repository": XLSR_REPO,
    "prompt_dim": PROMPT_DIM,
    "num_prompt_tokens": NUM_PROMPT_TOKENS,
    "num_wavelet_tokens": NUM_WAVELET_TOKENS,
    "prompt_dropout": PROMPT_DROPOUT,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "gpu": GPU_NAME,
    "best_checkpoint_format": "plain_state_dict",
}
CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

if start_epoch >= EPOCHS:
    print(f"이미 {start_epoch} epoch까지 완료되어 EPOCHS={EPOCHS} 이상입니다. 학습을 건너뜁니다.")
else:
    for epoch in range(start_epoch, EPOCHS):
        epoch_start = time.time()
        train_metrics = train_one_epoch(epoch)
        val_metrics = evaluate()
        elapsed_minutes = (time.time() - epoch_start) / 60.0

        record = {
            "epoch": epoch + 1,
            "train_loss": float(train_metrics["loss"]),
            "train_accuracy": float(train_metrics["accuracy"]),
            "val_loss": float(val_metrics["loss"]),
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_roc_auc_fake": float(val_metrics["roc_auc_fake"]),
            "val_eer": float(val_metrics["eer"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "minutes": elapsed_minutes,
        }
        history.append(record)

        improved = val_metrics["eer"] < best_eer
        if improved:
            best_eer = val_metrics["eer"]
            epochs_without_improvement = 0
            # 제출 생성기의 weights_only=True 로더와 바로 호환되는 순수 state_dict
            atomic_torch_save(model.state_dict(), BEST_PATH)
            print(f"best.pt 갱신: EER={best_eer:.6f}")
        else:
            epochs_without_improvement += 1

        resume_payload = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_eer": best_eer,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
            "config": run_config,
        }
        atomic_torch_save(resume_payload, LAST_CHECKPOINT_PATH)
        HISTORY_PATH.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")

        print(json.dumps(record, ensure_ascii=False, indent=2))
        print("best EER:", f"{best_eer:.6f}")

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"검증 EER가 {EARLY_STOPPING_PATIENCE} epoch 동안 개선되지 않아 조기 종료합니다.")
            break

if not BEST_PATH.is_file():
    raise FileNotFoundError("best.pt가 생성되지 않았습니다. 학습 로그를 확인하세요.")
print("학습 단계 완료:", BEST_PATH)


In [ ]:
# 선택 사항: 학습 곡선 확인
import matplotlib.pyplot as plt

if HISTORY_PATH.is_file():
    plotted_history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
else:
    plotted_history = history

if plotted_history:
    epochs_axis = [row["epoch"] for row in plotted_history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs_axis, [row["train_loss"] for row in plotted_history], marker="o", label="train")
    axes[0].plot(epochs_axis, [row["val_loss"] for row in plotted_history], marker="o", label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs_axis, [row["val_eer"] for row in plotted_history], marker="o")
    axes[1].set_title("Validation EER (lower is better)"); axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)
    axes[2].plot(epochs_axis, [row["val_roc_auc_fake"] for row in plotted_history], marker="o")
    axes[2].set_title("Validation ROC-AUC (FAKE)"); axes[2].set_xlabel("Epoch"); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## 5. `best.pt` 제출 생성기 호환성 검사

저장된 파일을 다시 읽어 현재 모델에 `strict=True`로 로드합니다. 프롬프트 텐서 크기, 전체 키 일치, class 0 FAKE 확률 계산, SHA-256을 확인한 뒤 제출 생성기에 붙여 넣을 정확한 경로를 출력합니다.


In [ ]:
def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

try:
    best_state = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
except TypeError:
    best_state = torch.load(BEST_PATH, map_location="cpu")

if not isinstance(best_state, dict) or not best_state:
    raise TypeError("best.pt가 비어 있거나 state_dict 형식이 아닙니다.")
if all(str(key).startswith("module.") for key in best_state):
    best_state = {str(key)[7:]: value for key, value in best_state.items()}

prompt_shape = tuple(best_state["wav2vec2_with_prompt.prompt_embedding"].shape)
wavelet_prompt_shape = tuple(best_state["wav2vec2_with_prompt.fprompt_embedding"].shape)
assert prompt_shape == (24, NUM_PROMPT_TOKENS, PROMPT_DIM)
assert wavelet_prompt_shape == (24, NUM_WAVELET_TOKENS, PROMPT_DIM)
model.load_state_dict(best_state, strict=True)
del best_state
gc.collect()

model.to(DEVICE)
model.eval()
example_audio, example_names, example_labels = next(iter(val_loader))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    _, logits = model(example_audio[:1])
    fake_probability = torch.softmax(logits.float(), dim=-1)[0, 0].item()

checkpoint_size_gib = BEST_PATH.stat().st_size / 2**30
checkpoint_hash = sha256(BEST_PATH)
print("strict=True 로드: OK")
print("prompt shape:", prompt_shape)
print("wavelet prompt shape:", wavelet_prompt_shape)
print("예시 class-0 FAKE 확률:", fake_probability)
print(f"best.pt 크기: {checkpoint_size_gib:.2f} GiB")
print("best.pt SHA256:", checkpoint_hash)
print("\n제출 패키지 생성기 설정 셀에 아래 한 줄을 넣으세요:\n")
print(f"WPT_CHECKPOINT = Path(r'{BEST_PATH}')")


### 완료 후 사용할 파일

Google Drive의 `MyDrive/WPT_XLSR_AASIST_FoR_20000/`에 다음 파일이 남습니다.

- `best.pt` — 학습된 전체 모델 체크포인트
- `last_checkpoint.pt` — Colab 세션이 끊긴 뒤 학습을 이어갈 때 사용하는 파일
- `training_config.json`, `training_history.json` — 재현 설정과 학습 기록
- `train_manifest.csv`, `val_manifest.csv` — 실제 선택된 표본 목록

아래 제출 생성 단계를 실행하면 같은 폴더에 Dacon 업로드용 `submit.zip`도 저장됩니다.


## 6. Dacon `submit.zip` 생성

Dacon 데이터 탭에서 받은 `open.zip`을 Google Drive의 `MyDrive/open.zip`에 둔 뒤 아래 셀을 실행합니다. `open.zip` 대신 그 안의 `baseline_submit.zip` 경로를 직접 지정해도 됩니다. 공식 baseline의 PANNs/HTDemucs 자산은 유지하고, DF-Arena 추론부만 이 노트북에서 학습한 WPT-XLSR-AASIST로 교체합니다.

생성물은 zip 최상위에 `model/`, `script.py`, `requirements.txt`만 포함하며, 평가 서버의 `data/test/`와 `data/sample_submission.csv`를 읽어 `output/submission.csv`를 생성합니다. 학습에 사용한 외부 데이터·모델·코드의 라이선스와 출처는 참가자가 최종 확인해야 합니다.


In [ ]:
import re
import zipfile

# Dacon 데이터 탭에서 받은 파일. 필요하면 이 한 줄만 수정하세요.
DACON_SOURCE_ZIP = Path('/content/drive/MyDrive/open.zip')
RUN_DUMMY_SMOKE_TEST = False

SUBMIT_WORK_DIR = WORK_DIR / 'dacon_submit_build'
SUBMIT_STAGE = SUBMIT_WORK_DIR / 'submit'
OPEN_EXTRACT_DIR = SUBMIT_WORK_DIR / 'open_extract'
SUBMIT_ZIP_PATH = OUTPUT_DIR / 'submit.zip'

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'안전하지 않은 zip 경로: {member.filename}')
        archive.extractall(destination)

def resolve_baseline_zip(source_zip):
    source_zip = Path(source_zip).expanduser().resolve()
    if not source_zip.is_file():
        raise FileNotFoundError(
            f'{source_zip} 파일이 없습니다. Dacon에서 open.zip을 내려받아 Drive에 올리거나 '
            'DACON_SOURCE_ZIP 경로를 수정하세요.'
        )
    if source_zip.name.lower() == 'baseline_submit.zip':
        return source_zip, None
    safe_extract(source_zip, OPEN_EXTRACT_DIR)
    matches = list(OPEN_EXTRACT_DIR.rglob('baseline_submit.zip'))
    if len(matches) != 1:
        raise FileNotFoundError(f'baseline_submit.zip을 정확히 하나 찾아야 합니다: {matches}')
    sample_files = list(OPEN_EXTRACT_DIR.rglob('sample_submission.csv'))
    dummy_root = sample_files[0].parent.parent if len(sample_files) == 1 else None
    return matches[0], dummy_root

def directory_size(path):
    return sum(item.stat().st_size for item in Path(path).rglob('*') if item.is_file())

if SUBMIT_WORK_DIR.exists():
    shutil.rmtree(SUBMIT_WORK_DIR)
SUBMIT_WORK_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_ZIP, DUMMY_ROOT = resolve_baseline_zip(DACON_SOURCE_ZIP)
print('공식 baseline:', BASELINE_ZIP)
print('학습 체크포인트:', BEST_PATH)


In [ ]:
# 1) 공식 baseline을 풀고 WPT 코드·XLS-R·학습 가중치를 포함합니다.
safe_extract(BASELINE_ZIP, SUBMIT_STAGE)
for required_name in ('model', 'script.py', 'requirements.txt'):
    if not (SUBMIT_STAGE / required_name).exists():
        raise FileNotFoundError(f'baseline에 {required_name}가 없습니다.')

df_arena_dir = SUBMIT_STAGE / 'model' / 'df_arena_1b'
if df_arena_dir.exists():
    shutil.rmtree(df_arena_dir)

WPT_MODEL_DIR = SUBMIT_STAGE / 'model' / 'wpt_xlsr_aasist'
WPT_CODE_DIR = WPT_MODEL_DIR / 'code'
WPT_XLSR_DIR = WPT_MODEL_DIR / 'xlsr_300m'
WPT_CODE_DIR.mkdir(parents=True, exist_ok=True)
WPT_XLSR_DIR.mkdir(parents=True, exist_ok=True)

for source_name in ('model.py', 'feature_extraction.py'):
    shutil.copy2(REPO_DIR / source_name, WPT_CODE_DIR / source_name)
(WPT_CODE_DIR / 'exp').mkdir(exist_ok=True)
shutil.copy2(REPO_DIR / 'exp' / 'feature_extraction_exp.py', WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py')
(WPT_CODE_DIR / 'exp' / '__init__.py').write_text('', encoding='utf-8')

# from_pretrained가 오프라인에서 필요한 XLS-R 설정과 가중치만 복사합니다.
for config_name in ('config.json', 'preprocessor_config.json'):
    source = XLSR_DIR / config_name
    if source.is_file():
        shutil.copy2(source, WPT_XLSR_DIR / config_name)
weight_candidates = [XLSR_DIR / 'model.safetensors', XLSR_DIR / 'pytorch_model.bin']
weight_source = next((path for path in weight_candidates if path.is_file()), None)
if weight_source is None:
    raise FileNotFoundError('XLS-R 모델 가중치를 찾지 못했습니다.')
shutil.copy2(weight_source, WPT_XLSR_DIR / weight_source.name)

# XLS-R backbone은 학습 중 동결되어 있으므로 base 가중치와 중복되는 키를 빼서 zip을 줄입니다.
try:
    full_state = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
except TypeError:
    full_state = torch.load(BEST_PATH, map_location='cpu')
if not isinstance(full_state, dict) or not full_state:
    raise TypeError('best.pt는 비어 있지 않은 state_dict여야 합니다.')
BACKBONE_PREFIX = 'wav2vec2_with_prompt.model.'
backbone_keys = [key for key in full_state if str(key).startswith(BACKBONE_PREFIX)]
if not backbone_keys:
    raise RuntimeError('best.pt에서 동결 XLS-R backbone 키를 찾지 못했습니다.')
submission_state = {key: value.cpu() for key, value in full_state.items() if key not in backbone_keys}
SUBMISSION_CHECKPOINT = WPT_MODEL_DIR / 'anti_spoofing_head.pt'
torch.save(submission_state, SUBMISSION_CHECKPOINT)
del full_state, submission_state
gc.collect()

# 원본 코드의 불필요한 선택 의존성·로그를 제거하고 DWT를 AMP 안전하게 유지합니다.
model_file = WPT_CODE_DIR / 'model.py'
model_text = model_file.read_text(encoding='utf-8')
model_text = model_text.replace('from pytorch_model_summary import summary\n', '')
model_text = model_text.replace('import torchvision.models as models\n', '')
model_file.write_text(model_text, encoding='utf-8')
feature_file = WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py'
feature_text = feature_file.read_text(encoding='utf-8')
feature_text = feature_text.replace('self.model.config.output_attentions = True', 'self.model.config.output_attentions = False')
feature_text = feature_text.replace("            print(hidden_state.shape,'hidden_state')\n", '')
wavelet_call = '        LL, band = self.dwt(x)'
wavelet_safe = (
    '        with torch.autocast(device_type=x.device.type, enabled=False):\n'
    '            LL, band = self.dwt(x.float())'
)
if wavelet_call in feature_text:
    feature_text = feature_text.replace(wavelet_call, wavelet_safe, 1)
elif wavelet_safe not in feature_text:
    raise RuntimeError('WaveletBlock DWT 호출부를 찾지 못했습니다.')
feature_file.write_text(feature_text, encoding='utf-8')
print('WPT 제출 자산 준비 완료:', f'{directory_size(WPT_MODEL_DIR) / 2**30:.2f} GiB')


In [ ]:
# 2) baseline script.py의 DF-Arena 추론부만 WPT-XLSR-AASIST로 교체합니다.
script_path = SUBMIT_STAGE / 'script.py'
script = script_path.read_text(encoding='utf-8')

old_path_block = 'DF_ARENA_DIR = MODEL_DIR / "df_arena_1b"'
new_path_block = '''WPT_DIR = MODEL_DIR / "wpt_xlsr_aasist"
WPT_CODE_DIR = WPT_DIR / "code"
WPT_XLSR_DIR = WPT_DIR / "xlsr_300m"
WPT_CHECKPOINT = WPT_DIR / "anti_spoofing_head.pt"'''
if old_path_block not in script:
    raise RuntimeError('baseline의 DF-Arena 경로 블록을 찾지 못했습니다.')
script = script.replace(old_path_block, new_path_block, 1)

wpt_section = r'''# -----------------------------------------------------------------------------
# 5. WPT-XLSR-AASIST를 이용한 성분별 Fake 추론
# -----------------------------------------------------------------------------

def load_wpt_model(device):
    if str(WPT_CODE_DIR) not in sys.path:
        sys.path.insert(0, str(WPT_CODE_DIR))
    from model import WPTW2V2AASIST

    model = WPTW2V2AASIST(
        model_dir=str(WPT_XLSR_DIR),
        prompt_dim=1024,
        device=device.type,
        sampling_rate=AUDIO_SAMPLE_RATE,
        num_prompt_tokens=6,
        num_wavelet_tokens=4,
        dropout=0.0,
        visual=False,
    )
    checkpoint = torch.load(WPT_CHECKPOINT, map_location="cpu", weights_only=True)
    if not isinstance(checkpoint, dict):
        raise TypeError("WPT checkpoint must contain a state_dict")
    incompatible = model.load_state_dict(checkpoint, strict=False)
    allowed_prefix = "wav2vec2_with_prompt.model."
    bad_missing = [key for key in incompatible.missing_keys if not key.startswith(allowed_prefix)]
    if bad_missing or incompatible.unexpected_keys:
        raise RuntimeError(
            f"WPT checkpoint mismatch. Missing={bad_missing[:10]}, "
            f"Unexpected={incompatible.unexpected_keys[:10]}"
        )
    model = model.to(device)
    model.eval()
    return model


def calculate_rms(audio):
    return float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))


def predict_fake(model, audio):
    if calculate_rms(audio) < SILENCE_RMS:
        return 0.0

    segments = [extract_segment(audio, start) for start in get_segment_starts(audio.size)]
    segment_scores = []
    for batch_start in range(0, len(segments), 2):
        batch_array = np.stack(segments[batch_start:batch_start + 2])
        batch_tensor = torch.from_numpy(batch_array)
        with torch.inference_mode(), torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()
        ):
            _, logits = model(batch_tensor)
        probabilities = torch.softmax(logits.float(), dim=-1)
        segment_scores.extend(probabilities[:, 0].cpu().tolist())

    return max(segment_scores)


'''
section_pattern = re.compile(
    r'# -+\n# 5\. DF-Arena 1B.*?(?=# -+\n# 6\.)',
    flags=re.DOTALL,
)
script, replacement_count = section_pattern.subn(wpt_section, script, count=1)
if replacement_count != 1:
    raise RuntimeError(f'DF-Arena 추론부 교체 횟수가 1이 아닙니다: {replacement_count}')

call_replacements = {
    'df_arena_model, fake_label_index = load_df_arena_model(device)':
        'wpt_model = load_wpt_model(device)',
    'voice_fake = predict_fake(\n            df_arena_model, fake_label_index, voice_audio, device\n        )':
        'voice_fake = predict_fake(wpt_model, voice_audio)',
    'music_fake = predict_fake(\n            df_arena_model, fake_label_index, music_audio, device\n        )':
        'music_fake = predict_fake(wpt_model, music_audio)',
}
for old_call, new_call in call_replacements.items():
    if old_call not in script:
        raise RuntimeError(f'baseline 호출부를 찾지 못했습니다: {old_call[:60]}')
    script = script.replace(old_call, new_call, 1)
script = script.replace('DF-Arena 1B', 'WPT-XLSR-AASIST')
if 'df_arena' in script.lower():
    raise RuntimeError('script.py에 DF-Arena 참조가 남아 있습니다.')
script_path.write_text(script, encoding='utf-8')

# 평가 서버 기본 패키지는 다시 설치하지 않고, 실제 추가 의존성만 설치합니다.
requirements = (
    '# Dacon server extras for WPT-XLSR-AASIST\n'
    'pytorch-wavelets==1.3.0\n'
    'PyWavelets==1.6.0\n'
)
(SUBMIT_STAGE / 'requirements.txt').write_text(requirements, encoding='utf-8')
print('script.py와 requirements.txt 교체 완료')


In [ ]:
# 3) 문법·구조·용량·무결성을 검사하고 Drive에 submit.zip을 생성합니다.
manifest = {
    'architecture': 'PANNs + HTDemucs + WPT-XLSR-AASIST',
    'training_dataset': KAGGLE_DATASET,
    'training_dataset_variant': DATASET_VARIANT,
    'wpt_repository': WPT_REPO,
    'wpt_commit': WPT_COMMIT,
    'best_checkpoint_sha256': sha256(BEST_PATH),
    'submission_checkpoint_sha256': sha256(SUBMISSION_CHECKPOINT),
    'xlsr_repository': XLSR_REPO,
    'note': '대회 제출 전 사용한 데이터, 코드, 모델의 라이선스와 출처를 참가자가 확인해야 합니다.',
}
(WPT_MODEL_DIR / 'SOURCE_MANIFEST.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

python_files = [
    SUBMIT_STAGE / 'script.py',
    WPT_CODE_DIR / 'model.py',
    WPT_CODE_DIR / 'feature_extraction.py',
    WPT_CODE_DIR / 'exp' / 'feature_extraction_exp.py',
]
for python_file in python_files:
    compile(python_file.read_text(encoding='utf-8'), str(python_file), 'exec')
    print('문법 OK:', python_file.relative_to(SUBMIT_STAGE))

if SUBMIT_ZIP_PATH.exists():
    SUBMIT_ZIP_PATH.unlink()
with zipfile.ZipFile(
    SUBMIT_ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED,
    compresslevel=1, allowZip64=True,
) as archive:
    for top_name in ('model', 'script.py', 'requirements.txt'):
        top_path = SUBMIT_STAGE / top_name
        if top_path.is_dir():
            for item in sorted(top_path.rglob('*')):
                if item.is_file() and '__pycache__' not in item.parts:
                    archive.write(item, item.relative_to(SUBMIT_STAGE).as_posix())
        else:
            archive.write(top_path, top_name)

with zipfile.ZipFile(SUBMIT_ZIP_PATH) as archive:
    names = archive.namelist()
    top_levels = {name.split('/', 1)[0] for name in names}
    if top_levels != {'model', 'script.py', 'requirements.txt'}:
        raise ValueError(f'잘못된 zip 최상위 구조: {top_levels}')
    if not any(name.startswith('model/') for name in names):
        raise ValueError('zip에 model 파일이 없습니다.')
    uncompressed_bytes = sum(info.file_size for info in archive.infolist())
    damaged_member = archive.testzip()
    if damaged_member:
        raise ValueError(f'손상된 zip 항목: {damaged_member}')

compressed_bytes = SUBMIT_ZIP_PATH.stat().st_size
if compressed_bytes > 10 * 2**30:
    raise ValueError(f'압축 크기가 10GB를 초과합니다: {compressed_bytes / 2**30:.2f} GiB')
if uncompressed_bytes > 32 * 2**30:
    raise ValueError(f'압축 해제 크기가 32GB를 초과합니다: {uncompressed_bytes / 2**30:.2f} GiB')

print('\nsubmit.zip 생성 완료:', SUBMIT_ZIP_PATH)
print(f'압축 크기: {compressed_bytes / 2**30:.2f} GiB')
print(f'압축 해제 크기: {uncompressed_bytes / 2**30:.2f} GiB')
print('submit.zip SHA256:', sha256(SUBMIT_ZIP_PATH))


In [ ]:
# 4) 선택 사항: open.zip의 더미 3개로 평가 서버와 같은 입출력 smoke test를 실행합니다.
if RUN_DUMMY_SMOKE_TEST:
    if DUMMY_ROOT is None or not (DUMMY_ROOT / 'data' / 'test').is_dir():
        raise FileNotFoundError('더미 테스트에는 data/test가 포함된 open.zip이 필요합니다.')
    smoke_packages = [
        'pytorch-wavelets==1.3.0', 'PyWavelets==1.6.0',
        'demucs==4.0.1', 'panns-inference==0.1.1', 'torchlibrosa==0.1.0',
        'librosa==0.10.2.post1', 'soxr>=0.3.2',
    ]
    run([sys.executable, '-m', 'pip', 'install', '-q', *smoke_packages])
    run([sys.executable, str(SUBMIT_STAGE / 'script.py')], cwd=DUMMY_ROOT)
    smoke_submission = DUMMY_ROOT / 'output' / 'submission.csv'
    if not smoke_submission.is_file():
        raise FileNotFoundError('output/submission.csv가 생성되지 않았습니다.')
    smoke_frame = pd.read_csv(smoke_submission)
    expected_columns = [
        'ID', 'FILE_FAKE_PROB', 'VOICE_FAKE_PROB', 'MUSIC_FAKE_PROB',
        'VOICE_PRESENT_PROB', 'MUSIC_PRESENT_PROB',
    ]
    assert list(smoke_frame.columns) == expected_columns
    assert len(smoke_frame) == 3
    assert smoke_frame[expected_columns[1:]].apply(lambda column: column.between(0, 1).all()).all()
    display(smoke_frame)
else:
    print('더미 smoke test 생략: 필요하면 RUN_DUMMY_SMOKE_TEST=True로 바꾸세요.')
    print('최종 제출 파일:', SUBMIT_ZIP_PATH)
